In [ ]:
%%writefile fmm.h
#pragma once           // il file viene incluso una sola volta
#include <cstdio>    
#include <vector>
#include <cmath>      // fminf, sqrtf
#include <queue>       // std::priority_queue
#include <functional>  // std::greater
#include <utility>     // std::pair
#include <chrono>      // misura dei tempi
#include <fstream>     // scrittura su file
const float INF = 1e30f;
typedef std::pair<float, int> Entry; // coppia (u, indice) per la coda di priorita'

__host__ __device__ float solve_quadratic(float a, float b, float F, float h)
{
    float Fh = F * h;
    if (fabsf(a - b) <= Fh)                                   
        return 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));
    return fminf(a, b) + Fh;                                  
}

float solve_node(const std::vector<float>& u, const std::vector<float>& F,
                 int i, int j, int N, float h)
{
    // a: min dei vicini lungo x (stessa riga, colonna j-1 e j+1)
    float left  = (j > 0)     ? u[i * N + (j - 1)] : INF;
    float right = (j < N - 1) ? u[i * N + (j + 1)] : INF;
    // b: min dei vicini lungo y (stessa colonna, riga i-1 e i+1)
    float down  = (i > 0)     ? u[(i - 1) * N + j] : INF;
    float up    = (i < N - 1) ? u[(i + 1) * N + j] : INF;
    // Per arrivare devi scorrere come un indice globale: u[i][j] = u[i * N + j]!

    float a = fminf(left, right);
    float b = fminf(down, up);

    int idx = i * N + j;
    float q = solve_quadratic(a, b, F[idx], h);
    return fminf(u[idx], q);                                  // return min(u_ij, u)
}
void fmm(std::vector<float>& u, const std::vector<float>& F, int N, float h, int src)
{
    enum { LONTANO = 0, ATTIVO = 1, NOTO = 2 };      // stato di ogni nodo
    std::vector<char> state(N * N, LONTANO);    // array di char NxN, inizializzato a LONTANO

    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;
    heap.push(Entry(u[src], src));                   // la sorgente e' il primo Trial
    state[src] = ATTIVO;

    const int di[4] = {  0, 0, -1, 1 };              // spostamenti verso i 4 vicini:
    const int dj[4] = { -1, 1,  0, 0 };              // sinistra, destra, sotto, sopra

    while (!heap.empty()) {                          // while A != vuoto
        Entry trial = heap.top();                    // Trial <- radice del min-heap
        heap.pop();                                  // Rimuovi Trial da A
        int x = trial.second;                        // indice del nodo Trial
        if (state[x] == NOTO || trial.first > u[x])  // coppia "scaduta": si ignora
            continue;
        state[x] = NOTO;                             // Aggiungi Trial a Set

        int i = x / N, j = x % N;                    // riga e colonna del Trial
        for (int k = 0; k < 4; ++k) {                // foreach x_nb adiacente a Trial
            int ni = i + di[k], nj = j + dj[k];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N) continue;   // fuori dal dominio
            int nb = ni * N + nj;   // indice del nodo vicino
            if (state[nb] == NOTO) continue;         // if x_nb non appartiene a Set

            float q = solve_node(u, F, ni, nj, N, h);
            if (q < u[nb]) {                         // u(x_nb) <- soluzione di g(x_nb)
                u[nb] = q;
                state[nb] = ATTIVO;                  // Aggiungi x_nb ad A
                heap.push(Entry(q, nb));
            }
        }
    }
}
double relative_error(const std::vector<float>& u, int N, float h)
{
    double num = 0.0, den = 0.0;
    for (int i = 0; i < N; ++i)
        for (int j = 0; j < N; ++j) {
            double x = -1.0 + j * h, y = -1.0 + i * h;   // coordinate del nodo (i, j)
            double U = std::sqrt(x * x + y * y);        // soluzione esatta
            double d = u[i * N + j] - U;
            num += d * d;                               // ||u - U||^2
            den += U * U;                               // ||U||^2
        }
    return std::sqrt(num / den);    // Norma euclidea relativa: ||u - U|| / ||U||
}

In [ ]:
%%writefile fim.cu
#include <cstdio>          
#include <cstdlib>         
#include <cuda_runtime.h>
#include "fmm.h"           // Fast Marching

// Controllo degli errori CUDA: se la chiamata fallisce, stampa file, riga e motivo ed esce.
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)
#define TILE 8 // dimensione del blocco 8x8 = 64 thread

__global__ void test_solve_kernel(float* res, float h)
{
    res[0] = solve_quadratic(0.0f, INF, 1.0f, h);            // vicino sull'asse:  a = 0, b = INF
    res[1] = solve_quadratic(h, h, 1.0f, h);                 // vicino diagonale:  a = b = h
}

int main()
{
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s | multiprocessori: %d | max thread per blocco: %d | shared memory per blocco: %zu KB\n",
           prop.name, prop.multiProcessorCount, prop.maxThreadsPerBlock, prop.sharedMemPerBlock / 1024);

    const int   N = 101;
    const float h = 2.0f / (N - 1);
    std::vector<float> u_fmm(N * N, INF);
    std::vector<float> F(N * N, 1.0f);
    const int si = N / 2, sj = N / 2;
    u_fmm[si * N + sj] = 0.0f;
    fmm(u_fmm, F, N, h, si * N + sj);   // CPU: Fast Marching sequenziale
    
    const float hp = 0.5f;
    float* d_res;
    CUDA_CHECK(cudaMalloc(&d_res, 2 * sizeof(float)));
    test_solve_kernel<<<1, 1>>>(d_res, hp);                  // 1 blocco da 1 thread
    CUDA_CHECK(cudaGetLastError());
    float res[2];
    CUDA_CHECK(cudaMemcpy(res, d_res, 2 * sizeof(float), cudaMemcpyDeviceToHost)); // copia da GPU a CPU
    CUDA_CHECK(cudaFree(d_res));
    printf("Algoritmo 2:  GPU asse = %.4f, diagonale = %.4f  |  CPU asse = %.4f, diagonale = %.4f\n",
           res[0], res[1], solve_quadratic(0.0f, INF, 1.0f, h), solve_quadratic(h, h, 1.0f, h));

    return 0;
}


In [ ]:
!nvcc -O3 -arch=sm_75 fim.cu -o fim && ./fim